# **WESAD Leave-One-Subject-Out (LOSO) Evaluation**

このノートブックでは、1D-CNNを用いてWESADデータセットのストレス検出をLOSO評価で行います。
LOSO評価は、1人の被験者をテスト用、残りを訓練用として繰り返す手法で、未知の被験者に対する汎化性能を正しく評価できます。

## **1. Imports & Global Parameters**

In [ ]:
import os
import pickle
import numpy as np
import pandas as pd
from scipy import stats
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import confusion_matrix, classification_report, f1_score, accuracy_score
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Conv1D, MaxPooling1D, BatchNormalization, Dropout, Flatten, Dense, Input, GlobalAveragePooling1D
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.utils import resample, shuffle
from sklearn.utils.class_weight import compute_class_weight

# Global parameters
DATASET_DIR = "/work/shinsaku-t/naist_reserch/self_learning_wesad/WESAD"
WINDOW_SECONDS = 5
FS = 700
WINDOW_SIZE = WINDOW_SECONDS * FS  # 3500
NUM_CHANNELS = 8  # ECG, EDA, EMG, Resp, Temp, ACCx, ACCy, ACCz

## **2. Data Segmentation and Utility Functions**

In [ ]:
def get_window(chest, label_arr, start, end):
    def get_signal(key):
        s = np.array(chest.get(key, np.zeros(WINDOW_SIZE))[start:end]).ravel()
        if len(s) < WINDOW_SIZE:
            s = np.pad(s, (0, WINDOW_SIZE - len(s)))
        return s

    try:
        ECG  = get_signal('ECG')
        EDA  = get_signal('EDA')
        EMG  = get_signal('EMG')
        Resp = get_signal('Resp')
        Temp = get_signal('Temp')
        acc  = np.array(chest.get('ACC', np.zeros((WINDOW_SIZE, 3)))[start:end])
        if acc.ndim == 2 and acc.shape[1] == 3:
            ACC_x, ACC_y, ACC_z = acc[:,0], acc[:,1], acc[:,2]
        else:
            ACC_x = ACC_y = ACC_z = np.zeros(WINDOW_SIZE)

        signals = [ECG, EDA, EMG, Resp, Temp, ACC_x, ACC_y, ACC_z]
        window_data = np.stack(signals, axis=-1)
        mode_result = stats.mode(label_arr[start:end], keepdims=True)
        mode_label = int(mode_result.mode[0])
        return window_data, mode_label
    except Exception as e:
        print(f"Error at window {start}: {e}")
        return None, None

def segment_data(data_dict):
    X, y = [], []
    if 'signal' not in data_dict or 'label' not in data_dict:
        return np.array([]), np.array([])
    chest = data_dict['signal']['chest']
    total_samples = len(next(iter(chest.values())))
    num_windows = total_samples // WINDOW_SIZE
    for i in range(num_windows):
        start = i * WINDOW_SIZE
        end = start + WINDOW_SIZE
        window_data, label = get_window(chest, data_dict['label'], start, end)
        if window_data is not None:
            # Filter and Map Labels (Binary Stress: 2 -> 1, others 1,3,4 -> 0)
            if label in [1, 2, 3, 4]:
                binary_label = 1 if label == 2 else 0
                X.append(window_data)
                y.append(binary_label)
    return np.array(X), np.array(y)

def build_model(input_shape):
    model = Sequential([
        Input(shape=input_shape),
        Conv1D(32, 7, activation='relu', padding='same'),
        BatchNormalization(),
        MaxPooling1D(2),
        Conv1D(64, 5, activation='relu', padding='same'),
        BatchNormalization(),
        MaxPooling1D(2),
        Conv1D(128, 3, activation='relu', padding='same'),
        BatchNormalization(),
        MaxPooling1D(2),
        GlobalAveragePooling1D(),
        Dropout(0.4),
        Dense(64, activation='relu'),
        Dropout(0.3),
        Dense(1, activation='sigmoid')
    ])
    model.compile(optimizer=Adam(5e-4), loss='binary_crossentropy', metrics=['accuracy'])
    return model

## **3. Pre-load and Scale All Subjects**

In [ ]:
subject_data = {}
subjects = sorted([s for s in os.listdir(DATASET_DIR) if os.path.isdir(os.path.join(DATASET_DIR, s))])

for subj in subjects:
    path = os.path.join(DATASET_DIR, subj)
    pkl_file = os.path.join(path, f"{subj}.pkl")
    if not os.path.exists(pkl_file):
        continue
    
    print(f"Processing Subject: {subj}")
    with open(pkl_file, 'rb') as f:
        data = pickle.load(f, encoding='latin1')
    
    X_sub, y_sub = segment_data(data)
    if X_sub.size > 0:
        # Scale subject data
        flat = X_sub.reshape(X_sub.shape[0], -1)
        scaled = StandardScaler().fit_transform(flat).astype(np.float32)
        X_scaled = scaled.reshape(-1, WINDOW_SIZE, NUM_CHANNELS)
        subject_data[subj] = (X_scaled, y_sub)

print(f"Loaded {len(subject_data)} subjects.")

## **4. LOSO Evaluation Loop**

In [ ]:
loso_accuracies = []
loso_f1_scores = []
results_log = []

all_subjects = sorted(list(subject_data.keys()))

for test_subj in all_subjects:
    print(f"\n--- Leave-One-Subject-Out: Testing on {test_subj} ---")
    
    # Split data
    X_test, y_test = subject_data[test_subj]
    
    X_train_list = []
    y_train_list = []
    
    for train_subj in all_subjects:
        if train_subj == test_subj:
            continue
        X_s, y_s = subject_data[train_subj]
        X_train_list.append(X_s)
        y_train_list.append(y_s)
    
    X_train = np.vstack(X_train_list)
    y_train = np.concatenate(y_train_list)
    
    # Shuffle training data
    X_train, y_train = shuffle(X_train, y_train, random_state=42)
    
    # Class weights
    class_weights = dict(enumerate(
        compute_class_weight('balanced', classes=np.unique(y_train), y=y_train)
    ))
    
    # Build and train model
    model = build_model((WINDOW_SIZE, NUM_CHANNELS))
    early_stop = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)
    
    model.fit(
        X_train, y_train,
        epochs=15,
        batch_size=64,
        validation_split=0.1,
        class_weight=class_weights,
        callbacks=[early_stop],
        verbose=0  # Set to 1 if you want to see training logs for each fold
    )
    
    # Evaluate
    y_probs = model.predict(X_test)
    y_pred = (y_probs > 0.5).astype(int)
    
    acc = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    
    loso_accuracies.append(acc)
    loso_f1_scores.append(f1)
    results_log.append({"Subject": test_subj, "Accuracy": acc, "F1-Score": f1})
    
    print(f"Result for {test_subj}: Accuracy = {acc:.4f}, F1-Score = {f1:.4f}")

print("\n--- LOSO Evaluation Summary ---")
print(f"Average Accuracy: {np.mean(loso_accuracies):.4f}")
print(f"Average F1-Score: {np.mean(loso_f1_scores):.4f}")

## **5. Visualize Results**

In [ ]:
df_results = pd.DataFrame(results_log)

plt.figure(figsize=(12, 6))
sns.barplot(data=df_results, x="Subject", y="Accuracy", palette="viridis")
plt.axhline(np.mean(loso_accuracies), color='red', linestyle='--', label=f'Mean Acc ({np.mean(loso_accuracies):.2f})')
plt.title("Accuracy by Subject (LOSO)")
plt.ylim(0, 1.05)
plt.legend()
plt.show()